# Python. Семинар 2. Условия, циклы и файлы вблизи


## 1. `and` и `or` возвращают не `True` и `False`

В лекции 1 было сказано, что правая часть логического выражения вычисляется
не всегда. Есть и вторая половина правила: результатом становится **один из
операндов**, а не логическое значение.


In [ ]:
"" or "аноним"

In [ ]:
"Иван" or "аноним"

`or` возвращает первый операнд, который считается истинным, а если такого нет —
последний. `and` действует зеркально: возвращает первый ложный операнд, иначе
последний.


In [ ]:
5 and 0, 0 and 5, None or 0

Отсюда распространённая запись «значение по умолчанию»: `имя or "аноним"`.
Работает она ровно до первого нуля.


In [ ]:
port = 0

port or 8080

Ноль — полноценное значение, но ложное, и `or` его выбросил. То же будет с
пустой строкой и пустым списком. Если ноль допустим, проверять нужно не
истинность, а отсутствие значения:


In [ ]:
port = 0

port if port is not None else 8080

Правило: `or` для значения по умолчанию годится, когда «пусто» и «ноль» — одно
и то же. Во всех остальных случаях нужна явная проверка на `None`.


## 2. `any` и `all`: пустой вход и мнимая ленивость

Обе функции разбирались в лекции. Здесь — два места, где они ведут себя не так,
как читается код.

**Первое: на пустом входе ответы несимметричны.**


In [ ]:
any([]), all([])

`all([])` истинна, потому что ни одного ложного элемента не нашлось: утверждение
«все элементы подходят» для пустого набора формально верно. `any([])` ложна по
той же логике — ни одного истинного не нашлось.

Ломается на этом проверка вида «все поля заполнены»: на пустой форме полей нет,
`all` возвращает истину, и форма проходит как заполненная. Если пустой вход
недопустим, его проверяют отдельно.

**Второе: `any` прекращает перебор на первом истинном элементе — но список
внутри неё считается целиком до вызова.** Проверим функцией, которая
рассказывает о себе:


In [ ]:
def check(x: int) -> bool:
    print("проверяю", x)
    return x > 0


any([check(x) for x in [1, -1, 5]])

Напечатались все три проверки, хотя ответ был ясен на первой. Включение в
квадратных скобках сначала строит список — вызывает `check` для каждого
элемента, — и только готовый список попадает в `any`. Останавливаться уже
нечему.

Убрать скобки нельзя было бы, если бы не генераторные выражения: без них список
внутри — единственный доступный нам способ. Как выглядит по-настоящему ленивый
вариант, разбирается в лекции 9; там же станет видно, что та же запись без
скобок вызывает `check` ровно один раз.

Практический вывод: пока проверка дешёвая, разницы нет. Как только внутри
обращение к файлу или долгий расчёт, список внутри `any` перестаёт быть
безобидным.


## 3. Почему цикл пропускает элементы

Лекция предупреждала: последовательность, по которой идёт перебор, менять
нельзя. Посмотрим, что именно происходит.


In [ ]:
items = [1, 2, 3, 4]
seen = []

for x in items:
    seen.append(x)
    if x % 2 == 0:
        items.remove(x)

seen, items

Тройка не попала в `seen`, хотя её никто не удалял. Цикл `for` по списку идёт
по позициям: 0, 1, 2 и так далее. Пока он стоял на позиции 1 (двойка), двойку
удалили — элементы справа сдвинулись влево. На следующем шаге цикл запросил
позицию 2, а там уже лежала четвёрка: тройка проскочила.

Способов два. Первый — не менять исходный список, а построить новый включением:


In [ ]:
items = [1, 2, 3, 4]

[x for x in items if x % 2 != 0]

Второй — идти по копии, а менять оригинал. Копию списка даёт полный срез:


In [ ]:
items = [1, 2, 3, 4]

for x in items[:]:
    if x % 2 == 0:
        items.remove(x)

items

Первый способ предпочтительнее: он короче и не требует помнить о копии. Второй
нужен, когда список обязан остаться тем же объектом, — почему это иногда важно,
разбирается в лекции 4.


## 4. `match`: имя в образце — это захват, а не сравнение

В лекции образцами были литералы и структуры. Стоит написать вместо литерала
имя переменной — и смысл меняется полностью.


In [ ]:
limit = 10
value = 7

match value:
    case 10:
        answer = "ровно десять"
    case limit:
        answer = "совпало с limit?"

answer, limit

Ветка сработала не потому, что `7` равно `limit`. Образец `case limit` значит
«подойдёт что угодно, и это что угодно записать в `limit`» — как `case _`,
только с именем. После `match` переменная `limit` равна семи: прежнее значение
затёрто.

Такой образец подходит ко всему и потому обязан быть последним: поставленный
раньше, он делает остальные ветки недостижимыми, и Python сообщает об ошибке
при разборе кода.

Сравнить со значением переменной позволяет `if` внутри ветки — ограничитель:


In [ ]:
limit = 10
value = 7

match value:
    case x if x == limit:
        answer = "совпало с limit"
    case _:
        answer = "не совпало"

answer, limit

Ограничитель проверяется после того, как образец подошёл, и может использовать
захваченные имена. Сравниваются же напрямую только литералы — числа, строки,
`True`, `False`, `None` — и имена с точкой вида `Модуль.ИМЯ`.


## 5. Файлы вблизи

### У открытого файла есть позиция чтения

Файл читается не «целиком в любой момент», а с текущей позиции. После `read`
позиция стоит в конце, и второе чтение возвращает пустую строку.


In [ ]:
with open("demo.txt", "w", encoding="utf-8") as file:
    file.write("первая\nвторая\n")

with open("demo.txt", encoding="utf-8") as file:
    first = file.read()
    second = file.read()

first, second

По той же причине `for line in file` после `read` не делает ни одного шага.
Позицию возвращает в начало метод `seek`:


In [ ]:
with open("demo.txt", encoding="utf-8") as file:
    whole = file.read()
    file.seek(0)
    lines = file.readlines()

whole, lines

### Кодировку указывают всегда

Без аргумента `encoding` Python берёт кодировку из настроек системы: на одной
машине UTF-8, на другой cp1251, и один и тот же код читает файл по-разному.
Поэтому в лекции `encoding="utf-8"` стоит в каждом вызове `open`.

Что бывает при несовпадении, видно и без файлов — кодируем в UTF-8, читаем
байты как cp1251:


In [ ]:
"ы".encode().decode("cp1251")

Если байты вообще не складываются в текст, декодирование завершается ошибкой.
Аргумент `errors` позволяет вместо этого подставить символ-заглушку:


In [ ]:
b"\xff\xfe".decode("utf-8", errors="replace")

### `strip` съедает больше, чем перевод строки

При построчном чтении каждая строка приходит с `\n` на конце, и в лекции его
убирал `strip`. Но `strip` без аргументов убирает **любые** пробельные символы с
обоих концов — включая ведущие отступы, которые могут быть частью данных:


In [ ]:
line = "    отступ важен\n"

repr(line.strip()), repr(line.rstrip("\n"))

Если значим только перевод строки, точнее писать `rstrip("\n")`.


In [ ]:
import os

os.remove("demo.txt")

## 6. Импорт: откуда берётся чужой код

Каждый файл `.py` — модуль. `import` выполняет его тело целиком и кладёт
результат в словарь `sys.modules`. Повторный импорт берёт готовое и тело заново
не выполняет — поэтому импортировать один модуль из десяти мест дёшево.

Проверим на модуле стандартной библиотеки, который ещё не загружен. Пометка
`# noqa` в ячейке отключает замечание линтера: обычно импорты собирают наверху,
но здесь важно показать состояние до и после.

In [ ]:
import sys

print("statistics" in sys.modules)

import statistics  # noqa: E402

print("statistics" in sys.modules, statistics.mean([1, 2, 3]))

Где Python ищет модуль — перечислено в `sys.path`. Первым в списке стоит каталог
**запускаемого файла**, а вовсе не тот, из которого вы запускали команду.

In [ ]:
sys.path[:2]

> **Типичная ошибка.** `ModuleNotFoundError: No module named 'mypackage'` при
> запуске `python3 mypackage/tools/run.py` из корня проекта. В `sys.path` попал
> каталог `mypackage/tools`, а корня проекта там нет — значит, и пакет
> `mypackage` не найден. Запускать нужно как модуль:
> `python3 -m mypackage.tools.run`, то есть точки вместо слешей и без `.py`.

Две формы импорта различаются тем, что попадает в ваше пространство имён:

In [ ]:
import math
from math import sqrt

print(math.sqrt(9), sqrt(9))

`import math` вносит одно имя — сам модуль, обращение идёт через точку.
`from math import sqrt` вносит имя `sqrt` и тем самым может затереть уже
существующее. Поэтому `from module import *` не пишут: неизвестно, какие имена
оно принесёт и что затрёт.